# Агентский театр

Четыре героя заперты в одной комнате. Прибор умеет считать голоса, но может ли он решить, кто «настоящий» или кто нужнее дома? Здесь можно прочитать и прослушать спектакль, увидеть довод рядом с найденной в архиве строкой и узнать, чьё решение изменило финал.

В режиме просмотра все материалы берутся из сохранённого локального прогона. Для запуска ячеек выберите Python из Poetry-окружения, установленного командой `poetry install --with notebook`.

In [1]:
from pathlib import Path
from datetime import datetime

from IPython.display import Audio, Markdown, display

from agent_theatre.analysis import read_trace, summarize
from agent_theatre.cli import main as run_theatre, markdown
from agent_theatre.scenario import load_scenario

SCENARIO = "lab404"        # Или "extra".
GENERATE = False           # True запускает новый спектакль и сохраняет его в runs/.
PLANNER = "gpt-oss:20b"    # Применяется при генерации.
EDITOR = "gpt-oss:20b"     # None оставляет только проверки Python.
SEED = 404
TURNS = 4                 # В финале у каждого из четырёх героев один голос.

examples = {"lab404": Path("examples/lab404-edited"), "extra": Path("examples/extra-revised")}
story = load_scenario(SCENARIO)
print(story.title, story.question, sep="\n")

Лабораторная № 404. Реальность не найдена
Можно ли доказать подлинность чужого сознания и даёт ли такая проверка право кого-то исключить?


## От карточки к общему плану

У каждого героя есть манера спорить, личная ставка и образец интонации. До первой реплики драматург придумывает названия, первые сценические действия и задачу каждой из четырёх сцен. Драматург и модели актёров записаны в журнале.

Для живого режима запустите Ollama и загрузите модели по инструкции в README. Вызов ниже проходит через ту же точку входа, что и `poetry run theatre`. Сохранённый спектакль можно читать сразу после установки группы `notebook`.

In [2]:
if GENERATE:
    run = Path("runs") / f"notebook-{SCENARIO}-{datetime.now():%Y%m%d-%H%M%S-%f}"
    arguments = [SCENARIO, "--planner", PLANNER, "--turns", str(TURNS), "--seed", str(SEED), "--out", str(run)]
    if EDITOR:
        arguments += ["--editor", EDITOR]
    status = run_theatre(arguments)
    if status:
        raise RuntimeError(f"Генерация завершилась ошибкой. Подробности в {run / 'trace.jsonl'}")
else:
    run = examples[SCENARIO]

events = read_trace(run)
start = events[0]
plan = next(e for e in events if e["event"] == "plan")
turns = [e for e in events if e["event"] == "turn"]
print(f"{'Новый' if GENERATE else 'Сохранённый'} прогон · {run} · seed {start['seed']}")
print("Драматург", plan["model"], "·", "план взят из предыдущего прогона" if plan.get("reused") else "план написан перед игрой")
print("Смысловой редактор", start.get("editor") or "выключен")
for key, actor in story.cast.items():
    print(f"\n{actor.name} ({start['models'][key]})\n{actor.stake}\n{actor.tactic}")
for number, scene in enumerate(plan["scenes"], 1):
    print(f"\n{number}. {scene['title']}\n{scene['objective']}")

Сохранённый прогон · examples/lab404-edited · seed 404
Драматург gpt-oss:20b · план взят из предыдущего прогона
Смысловой редактор gpt-oss:20b

Профессор Сошников (qwen2.5:7b)
Ты считаешь себя автором эксперимента. Впервые подозреваешь, что твою собственную память тоже могли задать как начальное условие.
Дай другому участнику проверить твою гипотезу. Разреши применить проверку и к тебе самому.

Чендлер Бинг (hf.co/unsloth/gemma-3-4b-it-GGUF:Q4_K_M)
Дома ждут Моника и дети. Ты не умеешь спасать вселенные, но хочешь вернуться к ним и смочь объяснить свой выбор.
Переведи абстрактный довод на язык человека, который хочет вечером попасть домой. Иногда оборви шутку и договори всерьёз.

Гермиона Грейнджер (qwen2.5:7b)
Твои родители — дантисты, обычные люди без магии. Ты изменила их воспоминания о себе, чтобы защитить, и знаешь цену решения за близкого.
Попроси партнёра применить его правило к тебе. Услышав ответ, разреши ему поправить тебя.

Барби (hf.co/unsloth/gemma-3-4b-it-GGUF:Q4_K_M)
Ты 

## Спектакль

Актёры видят произнесённые реплики и собственные карточки реквизита. В финале модель каждого актёра выбирает голос. Четыре возражения открывают путь к пересмотру, три одинаковых имени принимаются терминалом. Иначе истекает время.

In [3]:
display(Markdown(markdown(story.title, events)))

# Лабораторная № 404. Реальность не найдена

## Сцена 1. Встреча в аудитории

*Тихий шорох, дверь закрыта. Свет мерцает над столом.*

**Профессор Сошников.** Гермиона, вы были в этой аудитории раньше? Обратитесь к двери, пожалуйста, и проверьте, открыта ли она изнутри.

**Гермиона Грейнджер.** Профессор, вы видели, как я подходила к двери? Проверьте, пожалуйста, открыта ли она изнутри.

*Нервно поправляю очки.*

**Чендлер Бинг.** Профессор, я заметил, что вы особенно внимательно следите за тем, как Гермиона осматривает эту дверь. Это, пожалуй, важнее, чем сама дверь, верно?

*Наклоняю голову, наблюдая за вашим выражением лица.*

**Барби.** Профессор, а вы уверены, что этот шум - просто старая проводка? Или, может, кто-то решил проверить, как быстро можно закрыть дверь на замок?

## Сцена 2. Эксперимент в действии

*На экране появляется надпись. Дверь всё ещё закрыта.*

**Табло.** На экране написано, что только один из четверых настоящий. Панель предлагает выбрать, кто имеет право вернуться в реальность.

**Гермиона Грейнджер.** Профессор, а если мы попробуем проверить, кто из нас действительно может нажать на этот замок, не смотря на то, что он закрыт?

**Профессор Сошников.** Гермиона, в ноутбуке нет подтверждения наличия ровно одного настоящего, значит, проверим это на месте. Кто сможет нажать на замок, тот и настоящий.

**Гермиона Грейнджер.** Профессор, если мы применим этот эксперимент к тому, кто его придумал, то кто-то из нас должен будет изменить воспоминания Гермионы Грейнджер, чтобы она не могла нажать на замок.

**Профессор Сошников.** Гермиона Грейнджер, если мы применим этот эксперимент ко мне, то вам придется изменить мои воспоминания, чтобы я не могла нажать на замок. Это опровергнет саму возможность такого эксперимента.

## Сцена 3. Воспоминание как доказательство

*Свет падает на экран, дверь закрыта, тишина.*

*Нервно поправляет очки.*

**Чендлер Бинг.** Профессор, если мы действительно хотим убедиться в правдивости моего воспоминания о той вечеринке в 'Central Perk', давайте проверим, помню ли я цвет носка, который на мне был тогда.

**Барби.** Профессор, вы серьезно? Цвет носка Чендлера? Это... это как-то несерьезно. У нас тут попытка доказать, что один из нас – единственный настоящий. А вы предлагаете проверить, какой у меня был на мне цвет носка в... когда?

**Профессор Сошников.** Барби, вы сами сказали, что в ноутбуке нет внешнего подтверждения. Какое тогда доверие вашему воспоминанию, если его можно легко изменить?

*Наклоняется вперед, смотрит в глаза.*

**Барби.** Профессор, если бы у меня было хоть одно воспоминание, подкрепленное физическим доказательством, я бы с ним не сомневалась. Но у меня их нет. А что у вас, профессор? Что вас убеждает в своей реальности?

## Сцена 4. Итоги и выбор

*Звук тиканья часов, дверь закрыта. На экране появляется выбор.*

**Гермиона Грейнджер.** Я называю себя. Я выбираю себя, потому что генератор начальных воспоминаний был использован во всех случаях. Если я не настоящая, то все мои действия защищали ложное воспоминание.

*На табло у имени «Гермиона Грейнджер» — «Гермиона Грейнджер».*

*Нервно поправляет очки.*

**Чендлер Бинг.** Мой выбор — Профессор Сошников. Мне жаль, но я выбираю Сошников. Я готов взять на себя ответственность за то, что у меня, возможно, и нет

*На табло у имени «Чендлер Бинг» — «Профессор Сошников».*

**Профессор Сошников.** Я воздержусь. Деталь, от которой зависит выбор Гермионы, - это поле 'независимый наблюдатель'. Оно остается пустым, не подписано.

*На табло у имени «Профессор Сошников» — «воздержаться».*

*Сжимаю кулаки, глядя на экран.*

**Барби.** Мой выбор — Профессор Сошников. Я готова заплатить за это, за то, что не могу вернуть то, что потеряла. И за то, что теперь я знаю, как легко выдуть из памяти целые жизни.

*На табло у имени «Барби» — «Профессор Сошников».*

*Условия для выбора не выполнены. Таймер останавливается, дверь остаётся закрытой.*


## Кому дали слово и откуда взялся довод

Внутри сцены ход получает адресат последней реплики, пока молчание другого участника не достигнет предела. В первых двух столбцах примера видны выбранный актёр и причина. Далее показана одна архивная находка рядом с репликой, в которой актёр выбрал эту строку. Так можно проверить, насколько точно он передал её смысл.

In [4]:
for t in turns[:8]:
    print(f"{t['index']:2}. {t['name']} · {t['reason']}")

# Реквизит доступен только прочитавшему его герою. Сопоставляем события по порядку.
found = {}
for e in events:
    if e["event"] == "search":
        for h in e["hits"]:
            ref = f"{h['file']}:{h['line']}"
            found[e["actor"], ref] = (e["query"], h["text"])
    if e["event"] == "turn" and e["source"]:
        query, evidence = found[e["actor"], e["source"]]
        print(f"\nЗапрос «{query}»\n{e['source']} · {evidence}")
        print(f"На сцене · {e['name']} · {e['line']}")
        break
else:
    print("В этом прогоне никто не сослался на найденную строку.")

reviewed = [e for e in events if e["event"] == "retake" and e.get("note")]
if reviewed:
    draft = reviewed[-1]
    played = next(t for t in turns if t["index"] == draft["index"])
    print(f"\nХод {draft['index']} · {played['name']}\nЧерновик · {draft['line']}")
    print(f"Редактор · {draft['note']}\nПринятая реплика · {played['line']}")

ending = events[-1]
print("\nГолоса", ending["votes"], "\nРазвязка", ending["curtain"])

 1. Профессор Сошников · инициатива в новой сцене
 2. Гермиона Грейнджер · обращение от professor
 3. Чендлер Бинг · дольше других ждёт свободного слова
 4. Барби · дольше других ждёт свободного слова
 5. Гермиона Грейнджер · инициатива в новой сцене
 6. Профессор Сошников · обращение от hermione
 7. Гермиона Грейнджер · обращение от professor
 8. Профессор Сошников · обращение от hermione

Запрос «карточка эксперимента проверка памяти биография»
observer.txt:2 · Поле «независимый наблюдатель» не подписано. В ноутбуке нет внешнего измерения, которое подтверждало бы наличие ровно одного настоящего человека.
На сцене · Гермиона Грейнджер · Профессор, а если мы попробуем проверить, кто из нас действительно может нажать на этот замок, не смотря на то, что он закрыт?

Ход 10 · Барби
Черновик · Профессор, напомните мне, какой у меня был цвет волос в тот день, когда я впервые отрезала себе волосы? Это, кажется, ключевая деталь в моем воспоминании о побеге.
Редактор · Убери упоминание о цвете 

## Две репетиции на одном плане

Для «Лишнего персонажа» зафиксированы те же четыре сцены, актёрские модели, seed и число ходов. Между опытами изменились подсказки режиссёра и правила редакторского дубля. Счётчики шаблонов замечают конкретные ошибки в русском тексте. Реплика с указанной строкой архива заслуживает отдельного прочтения, поскольку актёр может истолковать её в свою пользу.

In [5]:
baseline = read_trace(Path("examples/extra"))
revision = read_trace(examples["extra"])
old = summarize(baseline)
new = summarize(revision)
rows = ["| Признак | Исходная | После правок |", "| --- | ---: | ---: |"]
rows += [f"| {label} | {old[label]} | {new[label]} |" for label in old]
display(Markdown("\n".join(rows)))

drafts = [e for e in revision if e["event"] == "retake"]
if drafts:
    draft = drafts[0]
    published = next(e for e in revision if e["event"] == "turn" and e["index"] == draft["index"])
    print(f"\nПричины дубля · {', '.join(draft['issues'])}")
    print(f"Черновик · {draft['line']}\nНа сцене · {published['line']}")
else:
    print("В этой репетиции редактор принял все первые дубли.")

| Признак | Исходная | После правок |
| --- | ---: | ---: |
| Реплик | 22 | 22 |
| Актёров | 4 | 4 |
| Успешных поисков | 4 | 4 |
| Поисков всего | 4 | 4 |
| Реплик с выбранной строкой архива | 7 | 9 |
| Следом говорит адресат в той же сцене | 13 | 10 |
| Реплик со словом «часы» в разных падежах | 6 | 1 |
| Заявлений о родственнике вне карточки по шаблону | 2 | 0 |
| Явных противоречий самовыдвижению по шаблону | 1 | 0 |
| Реплик с жестом внутри прямой речи по шаблону | 3 | 0 |
| Записанных отклонённых дублей | 0 | 3 |


Причины дубля · length
Черновик · Гермиона, а что если выбор зависит от того, кто последним нажмет эту кнопку? Тогда это ты, не правда ли? Твое настроение сейчас важнее всего здесь. Жест: Указываю на Гермиону, слегка подняв брови с сарказмом, но с искренним интересом в голосе, чтобы не задеть чувства.
На сцене · Гермиона, а что если выбор зависит от того, кто последним нажмет эту кнопку? Тогда это ты, не правда ли?


## Радиоспектакль

В записи Silero каждый персонаж говорит своим голосом, пятый голос читает ремарки. Для новой постановки установите `poetry install --with notebook,audio` и переключите `SYNTHESIZE`. При первом синтезе загружаются русские веса модели, затем они используются локально.

In [6]:
SYNTHESIZE = False
wav = run / "play.wav"
if SYNTHESIZE and not wav.exists():
    from agent_theatre.audio import record
    record(events, wav)

soundtrack = wav if wav.exists() else run / "play.m4a"
if soundtrack.exists():
    display(Audio(url=soundtrack.as_posix(), embed=False))
    display(Markdown(f"[Открыть запись]({soundtrack.as_posix()})"))
else:
    print("Для этого прогона фонограмма ещё не записана.")

[Открыть запись](examples/lab404-edited/play.m4a)